<a href="https://colab.research.google.com/github/buse-toklu/CENG467_Midterm/blob/main/question1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install -q transformers datasets evaluate accelerate scikit-learn nltk

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import random

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
# Load the SST-2 dataset from GLUE
dataset = load_dataset("glue", "sst2")

# Display dataset properties
print("--- SST-2 Dataset Properties ---")
print(dataset)
print("\nSample Data:")
print(f"Text: {dataset['train'][0]['sentence']}")
print(f"Label: {dataset['train'][0]['label']}")

# To keep training fast for this demonstration, we'll subsample the dataset
# Remove this in your final project for full evaluation
train_data = dataset['train'].shuffle(seed=42).select(range(5000))
val_data = dataset['validation']

train_texts = train_data['sentence']
train_labels = train_data['label']
val_texts = val_data['sentence']
val_labels = val_data['label']

In [ ]:
print("=== TASK 2: Preprocessing & Tokenization ===")

stop_words = set(stopwords.words('english'))

nltk.download('punkt_tab')

# Strategy 1: Word-level (For Sparse / TF-IDF)
# Decisions: Lowercasing, stopword removal, punctuation removal.
def preprocess_sparse(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and t not in string.punctuation]
    return " ".join(tokens)

train_texts_sparse = [preprocess_sparse(t) for t in train_texts]
val_texts_sparse = [preprocess_sparse(t) for t in val_texts]

# Strategy 2: Subword-level (For Contextual / Neural)
# Decisions: Minimal normalization to preserve context, uses WordPiece algorithm, explicit truncation.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_text = "The movie was not entirely terrible, but lacking."
print(f"Original Text: {sample_text}")
print(f"1. Sparse Preprocessing (Word-level): {preprocess_sparse(sample_text)}")
print(f"2. Contextual Preprocessing (Subword-level): {tokenizer.tokenize(sample_text)}")

In [ ]:
print("=== TASK 3 & 4: Classical Model (TF-IDF + LR) ===")

# Vectorization (Sparse Representation)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(train_texts_sparse)
X_val_tfidf = vectorizer.transform(val_texts_sparse)

# Model Training
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, train_labels)

# Evaluation
tfidf_preds = lr_model.predict(X_val_tfidf)
tfidf_acc = accuracy_score(val_labels, tfidf_preds)
tfidf_f1 = f1_score(val_labels, tfidf_preds, average='macro')

print(f"TF-IDF + LR -> Accuracy: {tfidf_acc:.4f} | Macro-F1: {tfidf_f1:.4f}")

In [ ]:
print("=== TASK 3 & 4: Neural Model (BiLSTM) ===")

# Encode sequences
def encode_for_lstm(texts, max_len=64):
    # Ensure texts is a standard Python list of strings
    texts = list(texts)
    encoded = tokenizer(texts, padding='max_length', truncation=True, max_length=max_len, return_tensors='pt')
    return encoded['input_ids']

X_train_seq = encode_for_lstm(train_texts)
X_val_seq = encode_for_lstm(val_texts)
y_train_seq = torch.tensor(train_labels)
y_val_seq = torch.tensor(val_labels)

class LSTMDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(LSTMDataset(X_train_seq, y_train_seq), batch_size=32, shuffle=True)

# Define BiLSTM Architecture (Dense Representation)
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, output_dim=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=tokenizer.pad_token_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(hidden)

bilstm_model = BiLSTM(tokenizer.vocab_size).to(device)
optimizer = Adam(bilstm_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training Loop
bilstm_model.train()
epochs = 3
for epoch in range(epochs):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = bilstm_model(batch_X.to(device))
        loss = criterion(predictions, batch_y.to(device))
        loss.backward()
        optimizer.step()

# Evaluation
bilstm_model.eval()
with torch.no_grad():
    preds = bilstm_model(X_val_seq.to(device))
    bilstm_preds = torch.argmax(preds, dim=1).cpu().numpy()

bilstm_acc = accuracy_score(val_labels, bilstm_preds)
bilstm_f1 = f1_score(val_labels, bilstm_preds, average='macro')

print(f"BiLSTM -> Accuracy: {bilstm_acc:.4f} | Macro-F1: {bilstm_f1:.4f}")

In [ ]:
print("=== TASK 3 & 4: Contextual Model (DistilBERT) ===")

def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=64)

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_val = val_data.map(tokenize_function, batched=True)

# Load metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1}

distilbert_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Train and Evaluate
trainer.train()
bert_results = trainer.evaluate()

bert_acc = bert_results['eval_accuracy']
bert_f1 = bert_results['eval_f1_macro']
print(f"DistilBERT -> Accuracy: {bert_acc:.4f} | Macro-F1: {bert_f1:.4f}")

In [ ]:
print("=== TASK 4: Error Analysis (DistilBERT) ===")

# Get predictions from the best model (usually DistilBERT)
predictions, labels, metrics = trainer.predict(tokenized_val)
predicted_classes = np.argmax(predictions, axis=1)

# Create a DataFrame to analyze misclassifications
val_df = pd.DataFrame({
    'text': val_texts,
    'true_label': labels,
    'predicted_label': predicted_classes
})

# Filter misclassified
misclassified = val_df[val_df['true_label'] != val_df['predicted_label']]
print(f"Total misclassified: {len(misclassified)} / {len(val_df)}\n")

# Extract 5 examples for analysis
sample_errors = misclassified.sample(n=5, random_state=42).copy()

label_map = {0: "Negative", 1: "Positive"}
sample_errors['true_label'] = sample_errors['true_label'].map(label_map)
sample_errors['predicted_label'] = sample_errors['predicted_label'].map(label_map)

pd.set_option('display.max_colwidth', None)
display(sample_errors)

# Template for your report (Do not run as code, just read the output and write your analysis)
print("\n--- Guidelines for Your Report ---")
print("Look at the 5 sentences above. Identify patterns such as:")
print("1. Complex Negation (e.g., 'not exactly a masterpiece')")
print("2. Sarcasm / Irony")
print("3. Mixed Sentiment (Starts positive, ends negative)")
print("4. Rare Idioms or Domain-Specific Jargon")

In [ ]:
print("\n" + "="*40)
print("       PERFORMANCE COMPARISON")
print("="*40)
print(f"1. Sparse (TF-IDF + LR):      {tfidf_acc:.4f}")
print(f"2. Dense (Word Embed + LSTM): {bilstm_acc:.4f}")
print(f"3. Contextual (DistilBERT):   {bert_acc:.4f}")
print("="*40)
print("""
Analysis:
- TF-IDF relies heavily on preprocessing (stopword removal). It usually performs worst because it ignores word order and context.
- BiLSTM captures sequence and local context, outperforming sparse methods, but requires training word embeddings from scratch (or loading pre-trained GloVe/Word2Vec).
- DistilBERT performs the best because its contextual representations dynamically understand polysemy and long-range dependencies, leveraging massive pre-training.
""")